# Argus VLM Optimization — Notebook 01: KV Cache Baseline & Quantization

**Goal:** Benchmark standard FP16 KV-cache vs quantized KV-cache (HQQ 4-bit, HQQ 2-bit, Quanto 4-bit) on single-image surveillance reasoning with Qwen2.5-VL-3B-Instruct.

### Colab Setup
Run the cell below to install dependencies if running on Google Colab.


In [ ]:
# Cell 1: Install Dependencies
# Run this on Google Colab (with GPU runtime)
!pip install -q torch torchvision transformers accelerate pillow pyyaml pandas matplotlib seaborn opencv-python-headless rouge-score
# Optional quantization backends:
!pip install -q hqq optimum quanto || true


In [ ]:
# Cell 2: Imports & Environment Check
import os
import sys
from pathlib import Path

# Add repository root to python path
repo_root = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

import torch
import pandas as pd
import yaml
from PIL import Image

from src.vlm.qwen_vlm import QwenVLMWrapper
from src.kv_cache.benchmark import KVCacheConfig, run_kv_cache_benchmark
from src.visualization.plots import plot_vram_vs_context

print(f"PyTorch Version: {torch.__version__}")
print(f"CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU Device: {torch.cuda.get_device_name(0)}")
    print(f"Initial VRAM Allocated: {torch.cuda.memory_allocated() / 1e9:.2f} GB")


In [ ]:
# Cell 3: Configuration
config_path = repo_root / "configs" / "kv_cache.yaml"
with open(config_path, "r") as f:
    kv_config = yaml.safe_load(f)

print("Loaded KV Cache Configurations:")
print(yaml.dump(kv_config, default_flow_style=False))

# Prepare experimental configurations list
configs_to_test = [
    KVCacheConfig(name="baseline_fp16_dynamic", backend="dynamic", nbits=16, description="Standard FP16 Dynamic Cache"),
    KVCacheConfig(name="quantized_hqq_4bit", backend="HQQ", nbits=4, description="HQQ INT4 Quantized Cache"),
    KVCacheConfig(name="quantized_hqq_2bit", backend="HQQ", nbits=2, description="HQQ INT2 Quantized Cache"),
    KVCacheConfig(name="quantized_quanto_4bit", backend="quanto", nbits=4, description="Quanto INT4 Quantized Cache"),
]


In [ ]:
# Cell 4: Model Loading
model_name = kv_config.get("model_name", "Qwen/Qwen2.5-VL-3B-Instruct")
torch_dtype = "bfloat16" if torch.cuda.is_bf16_supported() else "float16"

print(f"Loading {model_name}...")
try:
    wrapper = QwenVLMWrapper(
        model_name=model_name,
        torch_dtype=torch_dtype,
        device_map="auto" if torch.cuda.is_available() else "cpu"
    )
    print("VLM successfully loaded.")
except Exception as e:
    print(f"Model loading error (will run dummy mock for CPU testing if necessary): {e}")
    wrapper = None


In [ ]:
# Cell 5: Dataset / Input Loading
# Use a sample image or generate synthetic surveillance test frame
img_path = repo_root / "sample_data" / "surveillance_test.jpg"
if not img_path.exists():
    img = Image.new("RGB", (640, 480), color=(120, 130, 140))
    # Draw simple synthetic scene
    from PIL import ImageDraw
    d = ImageDraw.Draw(img)
    d.rectangle([100, 200, 200, 450], fill=(50, 50, 200)) # person/object
    d.rectangle([350, 250, 550, 400], fill=(180, 50, 50)) # vehicle
    img.save(img_path)
    print(f"Created synthetic surveillance test frame at {img_path}")
else:
    img = Image.open(img_path)

prompt = "Describe all detected objects, persons, actions, and potential safety concerns in this surveillance view."
print(f"Prompt: {prompt}")


In [ ]:
# Cell 6: Benchmark Execution
results = []
if wrapper is not None:
    results = run_kv_cache_benchmark(
        wrapper=wrapper,
        image=img,
        prompt=prompt,
        configs=configs_to_test,
        max_new_tokens=128
    )
else:
    print("Skipping live inference: Model wrapper unavailable.")


In [ ]:
# Cell 7: Save Results & Visualizations
output_csv = repo_root / "results" / "kv_cache" / "baseline_results.csv"
output_csv.parent.mkdir(parents=True, exist_ok=True)

if results:
    rows = [r.to_dict() for r in results]
    df = pd.DataFrame(rows)
    df.to_csv(output_csv, index=False)
    print(f"Results successfully saved to {output_csv}")
    print(df[["config", "backend", "nbits", "peak_memory_gb", "latency_seconds", "status"]])
else:
    print("No results to save.")
